In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from bs4 import BeautifulSoup
from urllib.request import urlopen, Request
import requests

In [14]:
url_list=[]
for i in range(1,11):
    url= f"https://www.flipkart.com/search?q=mobiles&as=on&as-show=on&otracker=AS_Query_TrendingAutoSuggest_1_0_na_na_na&otracker1=AS_Query_TrendingAutoSuggest_1_0_na_na_na&as-pos=1&as-type=HISTORY&suggestionId=mobiles&requestId=ef1a2283-0b9e-4824-888b-6bcd70f60143-3&page= {i}"
    url_list.append(url)

In [15]:
url_list

['https://www.flipkart.com/search?q=mobiles&as=on&as-show=on&otracker=AS_Query_TrendingAutoSuggest_1_0_na_na_na&otracker1=AS_Query_TrendingAutoSuggest_1_0_na_na_na&as-pos=1&as-type=HISTORY&suggestionId=mobiles&requestId=ef1a2283-0b9e-4824-888b-6bcd70f60143-3&page= 1',
 'https://www.flipkart.com/search?q=mobiles&as=on&as-show=on&otracker=AS_Query_TrendingAutoSuggest_1_0_na_na_na&otracker1=AS_Query_TrendingAutoSuggest_1_0_na_na_na&as-pos=1&as-type=HISTORY&suggestionId=mobiles&requestId=ef1a2283-0b9e-4824-888b-6bcd70f60143-3&page= 2',
 'https://www.flipkart.com/search?q=mobiles&as=on&as-show=on&otracker=AS_Query_TrendingAutoSuggest_1_0_na_na_na&otracker1=AS_Query_TrendingAutoSuggest_1_0_na_na_na&as-pos=1&as-type=HISTORY&suggestionId=mobiles&requestId=ef1a2283-0b9e-4824-888b-6bcd70f60143-3&page= 3',
 'https://www.flipkart.com/search?q=mobiles&as=on&as-show=on&otracker=AS_Query_TrendingAutoSuggest_1_0_na_na_na&otracker1=AS_Query_TrendingAutoSuggest_1_0_na_na_na&as-pos=1&as-type=HISTORY&sugg

In [16]:
headers={ 'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36'}

In [17]:
def create_html_page(url):
    response=requests.get(url,headers=headers)
    page_soup=BeautifulSoup(response.content,features='html')
    return page_soup

In [18]:
def create_final_data(page_soup):
    data={"Product Name":[],"Sale & Actual Price":[],"Review & Ratings":[],"Offer":[],"Camera":[],"Memory":[],"Battery":[],"Display":[],"Rating Number":[]}
    container=page_soup.find_all("div",{"class":"jIjQ8S"})
    for contain in container:
        pn=contain.find_all("div",{"class":"RG5Slk"})[0].text
        data["Product Name"].append(pn)
        
        rr=contain.find_all("div",{"class":"a7saXW"})
        if len(rr)==0:
            review_and_rating= None
            rating_number= None
        else:
            review_and_rating= rr[0].text
            rating_number= contain.find_all("div",{"class":"MKiFS6"})[0].text
        
        data["Review & Ratings"].append(review_and_rating)
        data["Rating Number"].append(rating_number)
        
        specification=contain.find_all("div",{"class":"CMXw7N"})[0]
        data["Memory"].append(specification.find_all('li')[0].text)
        data["Display"].append(specification.find_all('li')[1].text)
        data["Camera"].append(specification.find_all('li')[2].text)
        data["Battery"].append(specification.find_all('li')[3].text)
        
        price=contain.find_all("div",{"class":"oFEPlD"})[0].text
        data["Sale & Actual Price"].append(price)
        
        offer_list=contain.find_all("div",{"class":"HQe8jr"})
        if len(offer_list)==0:
            data["Offer"].append("No Offer")
        else:
            data["Offer"].append(offer_list[0].text)
    return data

# Main Section

In [19]:
data_frame_list=[]
for i, url in enumerate(url_list):
    print(f"started url page{i+1}")
    page_soup=create_html_page(url)
    data_info=create_final_data(page_soup)
    data_frame_list.append(data_info)

started url page1
started url page2
started url page3
started url page4
started url page5
started url page6
started url page7
started url page8
started url page9
started url page10


In [20]:
len(data_frame_list)

10

In [21]:
df_all=[]
for i in range(len(data_frame_list)):
    df=pd.DataFrame(data_frame_list[i])
    df_all.append(df)

In [22]:
final_df=pd.concat(df_all,ignore_index=True)
final_df

,Product Name,Sale & Actual Price,Review & Ratings,Offer,Camera,Memory,Battery,Display,Rating Number
0,"Samsung Galaxy F07 (Green, 64 GB)","₹9,999₹11,99916% off","4.36,152 Ratings & 367 Reviews",16% off,50MP + 2MP | 8MP Front Camera,4 GB RAM | 64 GB ROM | Expandable Upto 2 TB,5000 mAh Battery,17.02 cm (6.7 inch) HD+ Display,4.3
1,"realme P4 Lite (Sea Blue, 128 GB)","₹13,999","4.12,307 Ratings & 124 Reviews",No Offer,13MP Rear Camera | 5MP Front Camera,4 GB RAM | 128 GB ROM,6300 mAh Battery,17.14 cm (6.75 inch) HD+ Display,4.1
2,MOTOROLA Edge 60 Fusion 5G (PANTONE Slipstream...,"₹22,999₹24,9998% off","4.41,19,659 Ratings & 7,045 Reviews",8% off,50MP + 13MP | 32MP Front Camera,8 GB RAM | 128 GB ROM | Expandable Upto 1 TB,5500 mAh Battery,16.94 cm (6.67 inch) Display,4.4
3,"realme P4 Lite (Beach Gold, 128 GB)","₹13,999","4.12,307 Ratings & 124 Reviews",No Offer,13MP Rear Camera | 5MP Front Camera,4 GB RAM | 128 GB ROM,6300 mAh Battery,17.14 cm (6.75 inch) HD+ Display,4.1
4,"IQOO Z11x 5G (Prismatic Green, 128 GB)","₹21,750₹28,99924% off",4.3490 Ratings & 43 Reviews,24% off,50MP Rear Camera,6 GB RAM | 128 GB ROM,7200 mAh Battery,17.02 cm (6.7 inch) Display,4.3
...,...,...,...,...,...,...,...,...,...
235,"REDMI 15 5G (Frosted White, 128 GB)","₹19,990₹19,999","4.32,263 Ratings & 129 Reviews",No Offer,50MP Rear Camera,6 GB RAM | 128 GB ROM,7000 mAh Battery,17.53 cm (6.9 inch) Display,4.3
236,OPPO K13 5G with 7000mAh and 80W SUPERVOOC Cha...,"₹24,999","4.473,501 Ratings & 4,942 Reviews",No Offer,50MP + 2MP | 16MP Front Camera,8 GB RAM | 256 GB ROM,7000 mAh Battery,16.94 cm (6.67 inch) Display,4.4
237,"OPPO Reno14 5G (Forest Green, 512 GB)","₹47,999","4.53,939 Ratings & 359 Reviews",No Offer,50MP + 8MP + 50MP | 50MP Front Camera,12 GB RAM | 512 GB ROM,6000 mAh Battery,16.74 cm (6.59 inch) Display,4.5
238,"OPPO Reno14 5G (Pearl White, 256 GB)","₹44,999","4.53,939 Ratings & 359 Reviews",No Offer,50MP + 8MP + 50MP | 50MP Front Camera,12 GB RAM | 256 GB ROM,6000 mAh Battery,16.74 cm (6.59 inch) Display,4.5


In [23]:
final_df["Product Name"].unique().shape

(168,)

In [24]:
final_df.shape

(240, 9)

In [25]:
final_df.drop_duplicates()

,Product Name,Sale & Actual Price,Review & Ratings,Offer,Camera,Memory,Battery,Display,Rating Number
0,"Samsung Galaxy F07 (Green, 64 GB)","₹9,999₹11,99916% off","4.36,152 Ratings & 367 Reviews",16% off,50MP + 2MP | 8MP Front Camera,4 GB RAM | 64 GB ROM | Expandable Upto 2 TB,5000 mAh Battery,17.02 cm (6.7 inch) HD+ Display,4.3
1,"realme P4 Lite (Sea Blue, 128 GB)","₹13,999","4.12,307 Ratings & 124 Reviews",No Offer,13MP Rear Camera | 5MP Front Camera,4 GB RAM | 128 GB ROM,6300 mAh Battery,17.14 cm (6.75 inch) HD+ Display,4.1
2,MOTOROLA Edge 60 Fusion 5G (PANTONE Slipstream...,"₹22,999₹24,9998% off","4.41,19,659 Ratings & 7,045 Reviews",8% off,50MP + 13MP | 32MP Front Camera,8 GB RAM | 128 GB ROM | Expandable Upto 1 TB,5500 mAh Battery,16.94 cm (6.67 inch) Display,4.4
3,"realme P4 Lite (Beach Gold, 128 GB)","₹13,999","4.12,307 Ratings & 124 Reviews",No Offer,13MP Rear Camera | 5MP Front Camera,4 GB RAM | 128 GB ROM,6300 mAh Battery,17.14 cm (6.75 inch) HD+ Display,4.1
4,"IQOO Z11x 5G (Prismatic Green, 128 GB)","₹21,750₹28,99924% off",4.3490 Ratings & 43 Reviews,24% off,50MP Rear Camera,6 GB RAM | 128 GB ROM,7200 mAh Battery,17.02 cm (6.7 inch) Display,4.3
...,...,...,...,...,...,...,...,...,...
230,"REDMI 15 5G (Sandy Purple, 128 GB)","₹19,989₹20,9994% off","4.34,262 Ratings & 210 Reviews",4% off,50MP Rear Camera,8 GB RAM | 128 GB ROM,7000 mAh Battery,17.53 cm (6.9 inch) Display,4.3
231,"REDMI 15 5G (Midnight Black, 128 GB)","₹19,990₹19,999","4.32,263 Ratings & 129 Reviews",No Offer,50MP Rear Camera,6 GB RAM | 128 GB ROM,7000 mAh Battery,17.53 cm (6.9 inch) Display,4.3
232,"MOTOROLA Edge 70 Fusion (Pantone ORIENT BLUE, ...","₹26,999₹42,99937% off","4.47,746 Ratings & 841 Reviews",37% off,50MP + 13MP | 32MP Front Camera,8 GB RAM | 128 GB ROM,7000 mAh Battery,17.27 cm (6.8 inch) Super HD Display,4.4
237,"OPPO Reno14 5G (Forest Green, 512 GB)","₹47,999","4.53,939 Ratings & 359 Reviews",No Offer,50MP + 8MP + 50MP | 50MP Front Camera,12 GB RAM | 512 GB ROM,6000 mAh Battery,16.74 cm (6.59 inch) Display,4.5


In [26]:
final_df.to_csv("Flipkart_mobile_phones.csv")